# Issue #999: check the T4 forecast bracket against a measured tok/s

`benchmarks/run-t4-colab-free-tier.md` publishes a forecast of **288-423 tok/s** for
Llama-3.1-8B, NF4, streamed, on a free Colab T4. That bracket has never been checked
against a measured figure: the run behind it was held to a 4 GB process cap, and the
notebook quoted no tok/s on purpose.

This notebook measures it, under the ruling on #999 (option a):

- **No memory cap.** Nothing here calls `set_per_process_memory_fraction`, and the config
  has no `stream_vram_override`. Each measured run is a fresh `soup bench train` process.
- **The record's own config**, minus the cap: `quantization: 4bit`, `stream_layers: true`,
  `stream_buffers: 2`, `batch_size: 1`, `max_length: 256`, LoRA r=8 / alpha=16, fp16.
- **`soup bench train`** does the timing: warm-up steps left out, a validity check that
  the model trained while it was timed, and the SM clock sampled over the counted steps.
- **Two fixes from #2215, each applied only if the installed release lacks it.** TRL
  casts a 4-bit model's adapters to bf16 on every card, and 0.77.0's `bench train` breaks
  on that twice: it fingerprints the trainable parameters through numpy, which has no
  bf16, and it skips the step where `soup train` casts those adapters back to fp32 on an
  fp16 card, so on a T4 the GradScaler fails at the first step. The runner checks the
  installed code for each and, only where needed, applies the fix: a byte-level hash for
  the fingerprint (taken before the first step and after the last, outside the timed
  window), and the same `align_trainable_dtype_for_fp16` call `soup train` makes before
  training. With both, `bench train` measures the run `soup train` makes. Each run
  records which fixes were applied.
- **Two data shapes**, two runs each, interleaved:
  - **A**: the record's own 32 short rows (about 40 tokens each), like for like with #910.
  - **B**: 32 rows longer than 256 tokens, so every sequence is truncated to exactly 256.
    The bracket is a compute-bound bound, and a 40-token step at batch 1 is mostly
    spent streaming layers, so B is the bracket's fair test.
- The bracket is compared with **total tokens/s, padding included**, as ruled on #999.
  Supervised tokens/s and token utilisation are reported beside it as context.

**Runtime → Change runtime type → T4 GPU** before you start. Expect about an hour: most
of the first run is downloading the 16 GB checkpoint and sharding it to NF4. The last
cell downloads a zip of every report and log. Send that zip back; nothing else is needed.

## 1. Card, host and Python

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import psutil

assert (3, 11) <= sys.version_info[:2] < (3, 14), (
    f"Python {platform.python_version()}: soup-cli 0.77.0 declares >=3.11,<3.14"
)


def smi(query):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return out.stdout.strip() if out.returncode == 0 else None


GPU = smi("name")
assert GPU and "T4" in GPU, f"Got {GPU!r}. Runtime -> Change runtime type -> T4 GPU."

WORK = Path("/content/issue999")
WORK.mkdir(parents=True, exist_ok=True)

vm = psutil.virtual_memory()
ENVIRONMENT = {
    "gpu": GPU,
    "driver": smi("driver_version"),
    "vram_total_mib": smi("memory.total"),
    "power_limit_w": smi("power.limit"),
    "sm_clock_max_mhz": smi("clocks.max.sm"),
    "pcie_link_gen_max": smi("pcie.link.gen.max"),
    "pcie_link_width_max": smi("pcie.link.width.max"),
    "host_ram_total_bytes": vm.total,
    "host_ram_available_bytes_at_start": vm.available,
    "cpu_count": os.cpu_count(),
    "disk_free_bytes": shutil.disk_usage("/content").free,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "memory_cap": "none: no set_per_process_memory_fraction, no stream_vram_override",
    "pytorch_cuda_alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
}
for key, value in ENVIRONMENT.items():
    print(f"{key:36s} {value}")

## 2. Install soup-cli 0.77.0

0.77.0 is the latest release and carries `soup bench train` with its clock sampling.
Colab preinstalls an old `torchao`, which makes `peft` raise inside `get_peft_model`, so
it is removed. Versions are read in a fresh process, so they are what the runs will use.

In [ ]:
%pip uninstall -q -y torchao
%pip install -q "soup-cli[train]==0.77.0"

In [ ]:
_probe = r"""
import importlib.metadata as m, json, torch
names = ("soup-cli", "torch", "transformers", "peft", "trl", "bitsandbytes", "accelerate")
found = {}
for name in names:
    try:
        found[name] = m.version(name)
    except m.PackageNotFoundError:
        found[name] = None
found["torch_cuda"] = torch.version.cuda
found["cudnn"] = torch.backends.cudnn.version()
print(json.dumps(found))
"""
VERSIONS = json.loads(subprocess.run(
    [sys.executable, "-c", _probe], capture_output=True, text=True, check=True
).stdout)
assert VERSIONS["soup-cli"] == "0.77.0", VERSIONS
for key, value in VERSIONS.items():
    print(f"{key:14s} {value}")

## 3. Data and configs

**A** is `notebooks/proof-4gb.ipynb` section 5's data, unchanged. **B** has the same 32
prompts with answers well over 256 tokens; Soup truncates a row at `max_length`, so every
B sequence is exactly 256 tokens. The two configs differ only in `data.train`.

In [ ]:
TOPICS = [
    ("streaming", "Only a couple of decoder layers are resident at any moment."),
    ("NF4", "Four-bit weights make the host-side store about four times smaller."),
    ("LoRA", "The base is frozen, so it is read and never written."),
    ("VRAM", "Peak memory is bounded by one layer instead of by the model."),
]
rows_a = [
    {"messages": [
        {"role": "user", "content": f"Question {i}: tell me about {topic}."},
        {"role": "assistant", "content": answer},
    ]}
    for i in range(8)
    for topic, answer in TOPICS
]

SENTENCES = [
    "Each decoder layer is copied to the GPU just before it runs and released after.",
    "The host keeps the frozen base in pinned memory so copies can overlap compute.",
    "Four-bit NF4 weights are dequantised on the card inside each matrix multiply.",
    "The adapter is the only part of the model that the optimizer ever updates.",
    "Gradient checkpointing recomputes activations instead of keeping them resident.",
    "A T4 has no bf16 units, so the run computes in fp16 with a gradient scaler.",
    "The scaler skips a step when it sees an overflow and halves its scale.",
    "Two buffers let the next layer arrive while the current one is computing.",
    "The embedding and the head are large, so they get a slot of their own.",
    "Peak memory follows the largest layer, not the size of the whole model.",
    "Throughput depends on how much compute each streamed byte is used for.",
    "Short sequences leave the card waiting on the copy engine most of the time.",
    "Longer sequences give each copied layer more work and raise utilisation.",
    "The forecast divides the measured matrix rate by six times the parameter count.",
    "Real runs have landed between two thirds of that ceiling and all of it.",
    "Host memory bandwidth and the PCIe link both bound how fast layers arrive.",
]
rows_b = []
for i in range(8):
    for t, (topic, answer) in enumerate(TOPICS):
        start = (4 * i + t) % len(SENTENCES)
        body = " ".join(SENTENCES[(start + k) % len(SENTENCES)] for k in range(32))
        rows_b.append({"messages": [
            {"role": "user", "content": f"Question {i}: tell me about {topic}."},
            {"role": "assistant", "content": f"{answer} {body}"},
        ]})

CONFIG = """
base: NousResearch/Meta-Llama-3.1-8B-Instruct
task: sft
data:
  train: {train}
  max_length: 256
training:
  epochs: 1
  batch_size: 1
  lr: 0.0002
  logging_steps: 1
  quantization: 4bit
  stream_layers: true
  stream_buffers: 2
  lora:
    r: 8
    alpha: 16
output: ./out-8b
"""

CONFIGS = {}
for variant, rows in (("a", rows_a), ("b", rows_b)):
    data = WORK / f"train-{variant}.jsonl"
    data.write_text("\n".join(json.dumps(r) for r in rows), encoding="utf-8")
    cfg = WORK / f"soup-{variant}.yaml"
    cfg.write_text(CONFIG.format(train=data.name), encoding="utf-8")
    CONFIGS[variant] = cfg
print(CONFIG.format(train="train-<a|b>.jsonl"))

## 4. The runner

Each run is `soup bench train` in a fresh process. While it runs, the cell samples the
process's host RSS and the host's available RAM every second, and `nvidia-smi` logs the
SM clock, power, temperature, utilisation and PCIe link generation every second. Output
is printed live and saved to `<run>.log`.

The runs are launched as `python -c LAUNCHER bench train ...`, which is the `soup` entry
point plus the fingerprint check described in the introduction.

28 optimizer steps with 8 warm-up steps leaves 20 counted. The warm-up covers the fp16
GradScaler's back-off; a counted step with a non-finite `grad_norm` makes the report
`valid: false`, and such a report is kept and reported as it is.

In [ ]:
import re
import threading
import time

STEPS, WARMUP = 28, 8
PATCH_MARKER = "[issue999] applied fixes from #2215:"
LAUNCHER = f"""
import hashlib, inspect, sys
import torch
import soup_cli.bench.collector as collector
import soup_cli.bench.train_run as train_run

applied = []

def _digest(named_params):
    hasher = hashlib.sha256()
    for name, param in named_params:
        hasher.update(name.encode("utf-8"))
        raw = param.detach().cpu().contiguous().reshape(-1).view(torch.uint8)
        hasher.update(raw.numpy().tobytes())
    return hasher.hexdigest()

try:
    collector._digest([("probe", torch.zeros(1, dtype=torch.bfloat16))])
except TypeError:
    collector._digest = _digest
    applied.append("fingerprint")

if "align_trainable_dtype_for_fp16" not in inspect.getsource(train_run.run_bench_train):
    from soup_cli.trainer.sft import SFTTrainerWrapper
    from soup_cli.utils.mixed_precision import align_trainable_dtype_for_fp16

    _setup = SFTTrainerWrapper.setup

    def setup(self, *args, **kwargs):
        out = _setup(self, *args, **kwargs)
        targs = self.trainer.args
        align_trainable_dtype_for_fp16(
            self.trainer.model,
            fp16=getattr(targs, "fp16", False),
            bf16=getattr(targs, "bf16", False),
        )
        return out

    SFTTrainerWrapper.setup = setup
    applied.append("fp16-adapter-alignment")

if applied:
    print("{PATCH_MARKER}", ", ".join(applied), flush=True)
sys.argv[0] = "soup"
from soup_cli.cli import run
run()
"""
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
RUN_ENV = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTHONUNBUFFERED": "1",
    "COLUMNS": "200",
    "NO_COLOR": "1",
}


def run_bench(name, config, steps=STEPS, warmup=WARMUP):
    report = WORK / f"{name}.json"
    log = WORK / f"{name}.log"
    smi_log = open(WORK / f"{name}.smi.csv", "w")
    smi_proc = subprocess.Popen(
        ["nvidia-smi",
         "--query-gpu=timestamp,clocks.sm,power.draw,temperature.gpu,"
         "utilization.gpu,pcie.link.gen.current",
         "--format=csv,noheader,nounits", "-lms", "1000"],
        stdout=smi_log, stderr=subprocess.DEVNULL,
    )
    args = ["bench", "train", "--config", str(config), "--steps", str(steps),
            "--warmup", str(warmup), "-o", str(report)]
    cmd = [sys.executable, "-c", LAUNCHER, *args]
    started = time.time()
    proc = subprocess.Popen(cmd, cwd=WORK, env=RUN_ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    host = {"peak_rss_bytes": 0, "min_available_bytes": psutil.virtual_memory().available}
    done = threading.Event()

    def sample():
        root = psutil.Process(proc.pid)
        while not done.is_set():
            try:
                rss = root.memory_info().rss
            except psutil.Error:
                rss = 0
            host["peak_rss_bytes"] = max(host["peak_rss_bytes"], rss)
            host["min_available_bytes"] = min(
                host["min_available_bytes"], psutil.virtual_memory().available
            )
            done.wait(1.0)

    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()
    with open(log, "w", encoding="utf-8") as fh:
        for line in proc.stdout:
            line = ANSI.sub("", line)
            fh.write(line)
            print(line, end="")
    code = proc.wait()
    done.set()
    sampler.join()
    smi_proc.terminate()
    smi_proc.wait()
    smi_log.close()
    side = {
        "name": name, "config": config.name, "command": ["soup", *args],
        "fixes_applied": next(
            (line.split(PATCH_MARKER, 1)[1].strip().split(", ")
             for line in log.read_text(encoding="utf-8").splitlines()
             if line.startswith(PATCH_MARKER)),
            [],
        ),
        "exit_code": code,
        "wall_seconds": round(time.time() - started, 1), **host,
    }
    (WORK / f"{name}.host.json").write_text(json.dumps(side, indent=2))
    print(f"\n== {name}: exit {code}, {side['wall_seconds'] / 60:.1f} min, "
          f"peak host RSS {host['peak_rss_bytes'] / 1e9:.2f} GB, "
          f"min host available {host['min_available_bytes'] / 1e9:.2f} GB")
    return side

## 5. Prime: download and shard (not measured)

A short run whose only job is to download the checkpoint and write the NF4 shard cache
(`~/.soup/layer-stream`), so every measured run starts from the same warm state. Its
exit code does not matter: with 2 steps the GradScaler's back-off can make it
`valid: false`. Its log is kept anyway.

In [ ]:
prime = run_bench("prime", CONFIGS["a"], steps=2, warmup=1)

## 6. The measured runs

Interleaved A, B, A, B, each in a fresh process, so drift on the shared Colab host
lands on both variants instead of on one.

In [ ]:
ORDER = [("a1", "a"), ("b1", "b"), ("a2", "a"), ("b2", "b")]
RUNS = {}
for name, variant in ORDER:
    RUNS[name] = run_bench(name, CONFIGS[variant])

## 7. Results against the bracket

Each run's own pre-flight bracket is read from its log, next to the published
288-423. "x of low" is measured total tok/s divided by the bracket's low end: below 1
means the bracket over-predicted.

In [ ]:
import statistics

PUBLISHED = (288.0, 423.0)
BRACKET = re.compile(r"forecast\s+(\d+)-(\d+) tok/s")
SOURCE = re.compile(
    r"from ([\d.]+) TFLOPS measured on this card now using (\w+)(?: @ (\d+) MHz)?"
)


def session_bracket(text):
    found = BRACKET.search(text)
    source = SOURCE.search(text)
    if not found:
        return None
    return {
        "low": float(found.group(1)), "high": float(found.group(2)),
        "tflops": float(source.group(1)) if source else None,
        "dtype": source.group(2) if source else None,
        "probe_clock_mhz": int(source.group(3)) if source and source.group(3) else None,
    }


def smi_medians(path):
    cols = {"sm_mhz": [], "power_w": [], "temp_c": [], "util_pct": [], "pcie_gen": []}
    for line in path.read_text().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != 6:
            continue
        for key, raw in zip(cols, parts[1:]):
            try:
                cols[key].append(float(raw))
            except ValueError:
                pass
    return {k: (statistics.median(v) if v else None) for k, v in cols.items()}


ROWS = []
for name, variant in ORDER:
    report_path = WORK / f"{name}.json"
    if not report_path.exists():
        ROWS.append({"run": name, "variant": variant, "report": "missing"})
        continue
    report = json.loads(report_path.read_text())
    log = (WORK / f"{name}.log").read_text(encoding="utf-8")
    cfg = report["resolved_config"]["training"]
    seqs = (report["timing"]["counted_steps"] * cfg["batch_size"]
            * cfg["gradient_accumulation_steps"])
    total = report["throughput"]["total_tokens_per_second"]
    clock = report["provenance"]["sm_clock_mhz_busy"]
    own = session_bracket(log)
    ROWS.append({
        "run": name,
        "variant": variant,
        "valid": report["valid"],
        "failures": [f["check"] for f in report["failures"]],
        "fixes_applied": RUNS[name]["fixes_applied"],
        "counted_steps": report["timing"]["counted_steps"],
        "tokens_per_sequence": report["tokens"]["total"] / seqs if seqs else None,
        "total_tok_s": total,
        "supervised_tok_s": report["throughput"]["useful_tokens_per_second"],
        "utilisation": report["tokens"]["utilisation"],
        "median_step_s": report["timing"]["median_seconds"],
        "p95_step_s": report["timing"]["p95_seconds"],
        "sm_busy_min_med_max": (clock["min"], clock["median"], clock["max"]),
        "sm_samples": clock["sample_count"],
        "peak_alloc_gb": (report["memory"]["max_memory_allocated_bytes"] or 0) / 1e9,
        "peak_reserved_gb": (report["memory"]["max_memory_reserved_bytes"] or 0) / 1e9,
        "peak_host_rss_gb": RUNS[name]["peak_rss_bytes"] / 1e9,
        "min_host_available_gb": RUNS[name]["min_available_bytes"] / 1e9,
        "session_bracket": own,
        "x_of_published_low": total / PUBLISHED[0] if total else None,
        "x_of_published_high": total / PUBLISHED[1] if total else None,
        "x_of_session_low": total / own["low"] if total and own else None,
        "smi_medians_whole_run": smi_medians(WORK / f"{name}.smi.csv"),
    })

for row in ROWS:
    print(json.dumps(row, indent=1, default=str))

print("\nper variant (valid runs only):")
for variant in ("a", "b"):
    vals = [r["total_tok_s"] for r in ROWS
            if r.get("variant") == variant and r.get("valid") and r.get("total_tok_s")]
    if not vals:
        print(f"  {variant}: no valid run")
        continue
    mean = statistics.mean(vals)
    spread = (max(vals) - min(vals)) / mean if len(vals) > 1 else 0.0
    print(f"  {variant}: total tok/s {', '.join(f'{v:.1f}' for v in vals)}  "
          f"mean {mean:.1f}  spread {spread:.1%}  "
          f"= {mean / PUBLISHED[0]:.3f}x of 288, {mean / PUBLISHED[1]:.3f}x of 423")

SUMMARY = {
    "harness": "benchmarks/harness/issue999_t4_bracket_check.ipynb",
    "environment": ENVIRONMENT, "versions": VERSIONS,
    "steps": STEPS, "warmup": WARMUP, "published_bracket": PUBLISHED,
    "prime": prime, "runs": ROWS,
}
(WORK / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))

## 8. Download everything

In [ ]:
archive = shutil.make_archive("/content/issue999-results", "zip", WORK)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Not on Colab: copy the file above by hand.")